- In this notebook, you will learn how to load TEC data at temporal resolutions of 5 minutes and 30s from madrigal database.
<br>

- Data is available to download from: https://cedar.openmadrigal.org/list (make sure to cite the source).
<br>

- We will then make mlat-mlt as well as geographical lat-lon maps of vertical TEC /VTEC), and interpret the results.
<br>

- If you do not have any of the necessary (below) modules/libraries, you can install them using <font color="red">pip install library_name.</font> Example: pip install aacgmv2

# Load required libraries/modules


In [ ]:
import pandas as pd
import numpy as np

import os
import glob

import datetime as dt
import time
from datetime import datetime
from tqdm.auto import tqdm

import matplotlib
import matplotlib.pyplot as plt
from magpyplot import register_projections
register_projections()  # register MagPyPlot projections to Matplotlib
from matplotlib.pyplot import subplot, show
from mpl_toolkits.basemap import Basemap

import warnings
from scipy.interpolate import griddata
import aacgmv2
import pytz   # for getting utc from unixtime


import json
import cdflib
import h5py



# for coastlines
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cv2  # for making movies



# Give data directory paths

In [ ]:
# Enter date to create the TEC plots
date_here = input(
    "Enter date for plotting MLAT-MLT TEC plots in dd_mm_yyyy format (eg: 10_05_2024) : "
)

month_string = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sept", "Oct", "Nov", "Dec"]
year_here = date_here[6:10]
month_here = date_here[3:5]
month_string_here = month_string[int(month_here)-1]
day_here = date_here[0:2]


'''
#¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
                        # Change the data file path here
#¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
'''

main_folder = '/home/precision/Documents/B_REP/'  # change the data file path here
madrigal_tec_folder = os.path.join(main_folder, "Data/")


# Analysing 5min resolution TEC data

## Create TEC dataframe 


In [ ]:
os.chdir(madrigal_tec_folder)
madrigal_tec_file = glob.glob("gps*hdf5*")[0]

with h5py.File(madrigal_tec_file, 'r') as f:

    dataset = f['Data']['Table Layout'] 
    data_loaded = dataset[:]
    
    # Make a dataframe of the data
    madrigal_tec_DF_original = pd.DataFrame(data_loaded)
    
    
    # print("Column Titles:", dataset.dtype.names)
    
madrigal_tec_DF_original

## New madrigal TEC dataframe containing high latitude data (i.e.  for Lat>=50deg)


In [ ]:
# Adjust latitude here to cover smaller or larger geographical regions

madrigal_tec_DF = madrigal_tec_DF_original.query("gdlat>=50").reset_index(drop=True)

madrigal_tec_DF.keys()

## Adding a datetime column

In [ ]:
madrigal_tec_DF = madrigal_tec_DF.rename(columns={"min": "minute", "sec": "second"})

year_madrigal = madrigal_tec_DF.loc[:, "year"].values
month_madrigal = madrigal_tec_DF.loc[:, "month"].values
day_madrigal = madrigal_tec_DF.loc[:, "day"].values
hour_madrigal = madrigal_tec_DF.loc[:, "hour"].values
minute_madrigal = madrigal_tec_DF.loc[:, "minute"].values
second_madrigal = madrigal_tec_DF.loc[:, "second"].values

datetime_format_madrigal = [
    dt.datetime(
        year_madrigal[ddtt],
        month_madrigal[ddtt],
        day_madrigal[ddtt],
        hour_madrigal[ddtt],
        minute_madrigal[ddtt],
        second_madrigal[ddtt],
    )
    for ddtt in tqdm(range(0, len(second_madrigal)))
]

madrigal_tec_DF.insert(loc=6, column="datetime", value=datetime_format_madrigal)


##  Inserting columns for MLAT, MLON, MLT to the above madrigal TEC dataframe 


In [ ]:
madrigal_tec_DF["mlat"] = 0.0
madrigal_tec_DF["mlon"] = 0.0
madrigal_tec_DF["mlt"] = 0.0
madrigal_tec_DF

madrigal_tec_DF = madrigal_tec_DF.apply(pd.to_numeric, errors="ignore")


lat_loaded = madrigal_tec_DF.loc[:, "gdlat"].values
lon_loaded = madrigal_tec_DF.loc[:, "glon"].values

datetime_loaded = madrigal_tec_DF.loc[:, "datetime"].values
datetime_loaded = pd.to_datetime(datetime_loaded)

datetime_loaded_unique = np.unique(datetime_loaded)
# converting datetime from 2013-11-07T00:02:30 to 2013-11-07 00:02:30 format
datetime_loaded_unique = pd.to_datetime(datetime_loaded_unique)

for dd in tqdm(range(0, len(datetime_loaded_unique))):
    same_time_ind = np.where(datetime_loaded == datetime_loaded_unique[dd])[0]
    lat_loaded_array = lat_loaded[same_time_ind]
    lon_loaded_array = lon_loaded[same_time_ind]

    mlat, mlon, mlt = np.array(
        aacgmv2.get_aacgm_coord_arr(
            lat_loaded_array, lon_loaded_array, 350, datetime_loaded_unique[dd] # 350 is 350 km (IPP)
        )
    )

    madrigal_tec_DF.loc[same_time_ind, "mlat"] = mlat
    madrigal_tec_DF.loc[same_time_ind, "mlon"] = mlon
    madrigal_tec_DF.loc[same_time_ind, "mlt"] = mlt





## Getting required data for plotting

In [ ]:
# columns from madrigal_tec_DF
datetime_loaded_ALL = madrigal_tec_DF.loc[:, "datetime"].values
mlat_loaded_ALL = madrigal_tec_DF.loc[:, "mlat"].values
mlt_loaded_ALL = madrigal_tec_DF.loc[:, "mlt"].values
tec_loaded_ALL = madrigal_tec_DF.loc[:, "tec"].values

nan_ind = np.where(
    (np.isnan(mlat_loaded_ALL))
    | (np.isnan(mlt_loaded_ALL))
    | (np.isnan(tec_loaded_ALL))
)[0]
datetime_loaded_ALL = np.delete(datetime_loaded_ALL, nan_ind)
mlat_loaded_ALL = np.delete(mlat_loaded_ALL, nan_ind)
mlt_loaded_ALL = np.delete(mlt_loaded_ALL, nan_ind)
tec_loaded_ALL = np.delete(tec_loaded_ALL, nan_ind)



## Function to convert coastlines from geographic lat, lon to magnetic lat, lon, mlt


In [ ]:
def convert_geo_coastline_to_mag(geo, date, alt):
    def convert_to_mag(date, alt):
        for glon, glat in geo.coords:
            [mlat, mlon, mlt] = aacgmv2.get_aacgm_coord_arr(glat, glon, alt, date)

            yield mlt.item(), mlat.item()

    # Return geometry object
    return type(geo)(list(convert_to_mag(date, alt)))


## Making global TEC maps


In [ ]:
# ignoring pcolormesh cell edges warnings.
warnings.filterwarnings("ignore", category=UserWarning)

#date_only_string = date_here[6:10] + "-" + date_here[3:5] + "-" + date_here[0:2]
datetime_loaded_ALL_unique = np.unique(datetime_loaded_ALL)

#for ddtt in tqdm(
#    range(0, len(datetime_loaded_ALL_unique)), desc="Making global TEC maps: "):

for ddtt in tqdm(range(90, 91)): # Change number here to change the associated datetime of the TEC map

    """
    # ¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    # iterating through timestamps in TEC map files
    # ¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    """
    req_time_ind = np.where(datetime_loaded_ALL == datetime_loaded_ALL_unique[ddtt])[0]
    req_time = pd.to_datetime(datetime_loaded_ALL_unique[ddtt])

    datetime_loaded = datetime_loaded_ALL[req_time_ind]
    mlat_loaded = mlat_loaded_ALL[req_time_ind]
    mlt_loaded = mlt_loaded_ALL[req_time_ind]
    tec_loaded = tec_loaded_ALL[req_time_ind]

   


     

    """
    # ¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    #                       Finding mlat, mlt co-ordinates of coastlines
    # ¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    """

    # Read in the geometry object of the coastlines
    cc = cfeature.NaturalEarthFeature(
        "physical", "coastline", "110m", color="k", zorder=2.0
    )

    geo_mag = []
    alt = 300  # in Km
    for geo in cc.geometries():
        geo_mag.append(
            convert_geo_coastline_to_mag(
                geo, pd.to_datetime(datetime_loaded_ALL_unique[ddtt]), alt
            )
        )

    cc_mag = cfeature.ShapelyFeature(geo_mag, ccrs.PlateCarree(), color="k", zorder=2.0)

 

    """
    # ¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    #                                   Starting plotting of figure
    # ¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    """
    fig = plt.figure(figsize=(18, 8))
    ax0 = fig.add_gridspec(1, 1)
    ax1 = fig.add_subplot(ax0[0], projection="mlt_north")
    ax1.set_rlim(90, 60)
    ax1.set_rticks([90, 80, 70, 60])

    choose_cmap = 'turbo' #gist_ncar, jet, viridis, plasma

    # ax1.plot_night_patch() # optional dimming of the night half of the plot

    tec_loaded_filtered = tec_loaded
    # tec_loaded_filtered = signal.savgol_filter(tec_loaded, 10, 3)
    mlt_loaded_and_gnss_combined = mlt_loaded
    mlat_loaded_and_gnss_combined = mlat_loaded
    tec_loaded_and_gnss_combined = tec_loaded_filtered

    

    """
    # ¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    # Pcolormesh plot of madrigal tec map
    # ¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    """
    mlt_list = np.arange(0, 24 + 1, 0.24)
    for ii in range(0, len(mlt_list)):
        if mlt_list[ii] >= 24:
            mlt_list[ii] = abs(mlt_list[ii] - 24)
        else:
            continue

    grid_mlt_flatten_ALL = []
    grid_mlat_flatten_ALL = []

    for mm in range(0, len(mlt_list) - 1):
        if np.floor(mlt_list[mm]) == 23 and np.floor(mlt_list[mm + 1]) == 0:
            mlt_data_idx = np.where(
                (mlt_loaded_and_gnss_combined >= mlt_list[mm])
                & (mlt_loaded_and_gnss_combined <= 24)
            )[0]
            mlat_data_present_init = np.unique(
                np.round(mlat_loaded_and_gnss_combined[mlt_data_idx])
            )
            
            
            mlat_data_present = np.unique(
                np.hstack(
                    [
                        mlat_data_present_init,
                        mlat_data_present_init + 1,
                        mlat_data_present_init - 1,
                    ]
                )
            )

            grid_mlt, grid_mlat = np.meshgrid(
                np.linspace(mlt_list[mm], 24, 24 * 2), np.linspace(0, 90, 90 * 2)
            )
        else:
            mlt_data_idx = np.where(
                (mlt_loaded_and_gnss_combined >= mlt_list[mm])
                & (mlt_loaded_and_gnss_combined <= mlt_list[mm + 1])
            )[0]
            mlat_data_present_init = np.unique(
                np.round(mlat_loaded_and_gnss_combined[mlt_data_idx])
            )
            mlat_data_present = np.unique(
                np.hstack(
                    [
                        mlat_data_present_init,
                        mlat_data_present_init + 1,
                        mlat_data_present_init - 1,
                    ]
                )
            )

            grid_mlt, grid_mlat = np.meshgrid(
                np.linspace(mlt_list[mm], mlt_list[mm + 1], 24 * 2),
                np.linspace(0, 90, 90 * 2),
            )

        grid_mlt_shape = np.shape(grid_mlt)

        grid_mlt_flatten = grid_mlt.flatten()
        grid_mlat_flatten = grid_mlat.flatten()
        grid_mlat_floor = np.round(grid_mlat_flatten)

        for gg in range(0, len(grid_mlat_floor)):
            if grid_mlat_floor[gg] not in mlat_data_present:
                grid_mlat_flatten[
                    gg
                ] = 1e100  # giving a high value instead of nan since it doesnt work in griddata
                grid_mlt_flatten[gg] = 1e100
            else:
                continue
        grid_mlt_flatten_ALL.append(grid_mlt_flatten)
        grid_mlat_flatten_ALL.append(grid_mlat_flatten)
        
    

    for rr in range(0, len(grid_mlat_flatten_ALL)):
        grid_mlt_to_plot = np.reshape(
            grid_mlt_flatten_ALL[rr], (grid_mlt_shape[0], grid_mlt_shape[1])
        )
       
        grid_mlat_to_plot = np.reshape(
            grid_mlat_flatten_ALL[rr], (grid_mlt_shape[0], grid_mlt_shape[1])
        )

        grid_tec_to_plot = griddata(
            (mlt_loaded_and_gnss_combined, mlat_loaded_and_gnss_combined),
            tec_loaded_and_gnss_combined,
            (grid_mlt_to_plot, grid_mlat_to_plot),
            method="linear",
        )
        

        ax1.grid(
            False
        )  # just so that the below ax1.pcolormesh does not throw error of <Auto-removal of grids>
        # and grids have to removed manually .....blah blah blah
        img = ax1.pcolormesh(
            grid_mlt_to_plot,
            grid_mlat_to_plot,
            grid_tec_to_plot,
            cmap=choose_cmap,
            vmin=0,
            vmax=30,
            zorder=1,
        )
        ax1.grid(True, color="black", lw=0.5)  # bringing the grids back on

    """
    # ¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    # Plot coastlines here
    # ¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    """
    for geom in cc_mag.geometries():
        ax1.plot(*geom.coords.xy, color="k", linewidth=0.7, alpha=1)
        
    

    """
    # ¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    # Plot title settings
    # ¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    """

    col = plt.colorbar(img, fraction=0.046, pad=0.1)
    col.set_label("VTEC (TECU)", fontsize=15)

    plt.rcParams["font.size"] = 15
    plt.rcParams["font.weight"] = "normal"

    date_as_title = req_time.strftime("%d-%m-%Y")
    time_as_title = req_time.strftime("%H:%M:%S")
    datetime_as_title = date_as_title + "\n" + time_as_title + "\n\n"
    ax1.set_title(
        " ".join(datetime_as_title), fontweight="bold", fontsize=18, loc="center"
    )

     
 

# Analysing 30s (high) resolution TEC data

In [ ]:
date_here = input(
    "Enter date for analysing 30s resolution TEC maps (eg: 20_07_2023) : "
)

month_string = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sept", "Oct", "Nov", "Dec"]
year_here = date_here[6:10]
month_here = date_here[3:5]
month_string_here = month_string[int(month_here)-1]
day_here = date_here[0:2]




'''
#¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
# Searching for 1min LOS madgrial TEC file
#¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
'''
os.chdir(madrigal_tec_folder)
madrigal_tec_file = glob.glob("los*")[0]



## Function  "getTimeData "

In [ ]:
def getTimeData(madrigalFile, timeArr, unixTime):
    
    """
    getTimeData returns a numpy recarray of all the data in madrigalFile from a given time

     Inputs:
     madrigalFile - path to Madrigal LOS Hdf5 file
     timeArr - a numpy array of the times (ut1_unix) in the entire file.
     unixTime - the unix time to select
     
     
     Returns a numpy recarray of the subset of data from the file from that one site
     """
    with h5py.File(madrigalFile, 'r') as f:

        # get a list of all the indices with the right site
        time_indices = timeArr == unixTime
        timeData = f['Data']['Table Layout'][time_indices]
        return(timeData)

## Read the first timestamp data - takes some time !


In [ ]:
t = time.time()

os.chdir(madrigal_tec_folder)
# we only need to do this step once, so that getting data from other sites is faster
f = h5py.File(madrigal_tec_file, 'r')
timeArr = f['Data']['Table Layout']['ut1_unix']

# get the list of unique times from this array
times = np.unique(timeArr)
times_datetime_format_madr = [datetime.fromtimestamp(tt, tz=pytz.utc) for tt in times]
times_datetime_format_madr = np.array(times_datetime_format_madr)

f.close()


timeData = getTimeData(madrigal_tec_file, timeArr, times[0])
print('Took %f secs to get %i measurements from the first time' % 
      (time.time()-t, len(timeData)))



## Create 3 x 2 subplots of TEC maps


In [ ]:
fig = plt.figure(figsize=(28, 15))
ax0 = fig.add_gridspec(2, 3)
ax0.update(left=0.05, right=0.9, wspace=0.1, hspace=0.01)
# wspace = width spacing of subplots, hspace = height spacing between subplots

plt.rcParams["font.size"] = 24
plt.rcParams["font.weight"] = "normal"
common_fnt_size = 24
choose_cmap = 'turbo' # jet, gist_ncar


"""
#¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
# Intervals needed - adjust as needed
#¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
"""

t1_needed = (dt.datetime(int(year_here), int(month_here), int(day_here), 2, 0)).replace(
    tzinfo=dt.timezone.utc)

t2_needed = (dt.datetime(int(year_here), int(month_here), int(day_here), 3, 30)).replace(
    tzinfo=dt.timezone.utc)

t3_needed = (dt.datetime(int(year_here), int(month_here), int(day_here), 5, 0)).replace(
    tzinfo=dt.timezone.utc)

t4_needed = (dt.datetime(int(year_here), int(month_here), int(day_here), 6, 30)).replace(
    tzinfo=dt.timezone.utc)

t5_needed = (dt.datetime(int(year_here), int(month_here), int(day_here), 8, 0)).replace(
    tzinfo=dt.timezone.utc)

t6_needed = (dt.datetime(int(year_here), int(month_here), int(day_here), 9, 30)).replace(
    tzinfo=dt.timezone.utc)


required_datetimes = np.array([t1_needed, t2_needed, t3_needed, t4_needed, t5_needed, t6_needed])

for tt in tqdm(range(0, len(required_datetimes)), desc='Running all timestamps'):

    os.chdir(madrigal_tec_folder)
    
    idx1 = np.where(times_datetime_format_madr == required_datetimes[tt])[0][0]
    idx2 = np.where(times_datetime_format_madr == required_datetimes[tt]+dt.timedelta(seconds=30))[0][0]


    timeData1 = getTimeData(madrigal_tec_file, timeArr, times[idx1])
    timeData2 = getTimeData(madrigal_tec_file, timeArr, times[idx2])
    
    
    '''
    #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    ##                           Remove columns that are not needed
    #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    '''
    
    madrigal_tec_DF_temp1 = pd.DataFrame(timeData1)
    madrigal_tec_DF_temp2 = pd.DataFrame(timeData2)
    
    madrigal_tec_DF_temp1 = madrigal_tec_DF_temp1.drop(columns=['recno', 'kindat',
       'kinst', 'ut1_unix', 'ut2_unix', 'pierce_alt', 'gps_site', 'sat_id', 'gnss_type',
       'gdlatr', 'gdlonr', 'los_tec', 'dlos_tec', 'rec_bias', 'drec_bias'])
    
    madrigal_tec_DF_temp2 = madrigal_tec_DF_temp2.drop(columns=['recno', 'kindat',
       'kinst', 'ut1_unix', 'ut2_unix', 'pierce_alt', 'gps_site', 'sat_id', 'gnss_type',
       'gdlatr', 'gdlonr', 'los_tec', 'dlos_tec', 'rec_bias', 'drec_bias'])
    
    '''
    #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    ##  Choose required latitude and elevation range (i.e.  for Lat>=5deg, ELv>=30deg)
    #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    '''
    # madrigal_tec_DF = madrigal_tec_DF_temp.query('gdlat>=40 & gdlat<=60').reset_index(drop=True) 
    # madrigal_tec_DF_1 = madrigal_tec_DF_temp1.query('gdlat>=10 & elm>=30').reset_index(drop=True)
    madrigal_tec_DF_1 = madrigal_tec_DF_temp1.query('gdlat>=10').reset_index(drop=True)
    madrigal_tec_DF_2 = madrigal_tec_DF_temp2.query('gdlat>=10').reset_index(drop=True)
    
        
    '''
    #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    ##          Combine data from 0s and 30s together to get a 1min DF
    #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
    '''
    if ( (np.unique(madrigal_tec_DF_1['year']) == np.unique(madrigal_tec_DF_2['year']))&
         (np.unique(madrigal_tec_DF_1['month']) == np.unique(madrigal_tec_DF_2['month']))&
         (np.unique(madrigal_tec_DF_1['day']) == np.unique(madrigal_tec_DF_2['day']))&
         (np.unique(madrigal_tec_DF_1['hour']) == np.unique(madrigal_tec_DF_2['hour']))&
         (np.unique(madrigal_tec_DF_1['min']) == np.unique(madrigal_tec_DF_2['min']))&
         (np.unique(madrigal_tec_DF_1['sec'])==0)&
         (np.unique(madrigal_tec_DF_2['sec'])==30) 
       
       ):
        madrigal_tec_DF = pd.concat([madrigal_tec_DF_1, madrigal_tec_DF_2]).reset_index(drop=True)
        
        
        '''
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        ##                                 Insert datetime as a column
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        '''
        year_madrigal = madrigal_tec_DF.loc[:, "year"].values
        month_madrigal = madrigal_tec_DF.loc[:, "month"].values
        day_madrigal = madrigal_tec_DF.loc[:, "day"].values
        hour_madrigal = madrigal_tec_DF.loc[:, "hour"].values
        minute_madrigal = madrigal_tec_DF.loc[:, "min"].values
        second_madrigal = madrigal_tec_DF.loc[:, "sec"].values

        datetime_format_madrigal = [
            dt.datetime(
                year_madrigal[ddtt],
                month_madrigal[ddtt],
                day_madrigal[ddtt],
                hour_madrigal[ddtt],
                minute_madrigal[ddtt],
                second_madrigal[ddtt],
            )
            for ddtt in tqdm(range(0, len(second_madrigal)), desc='Adding datetime column', leave=False)
        ]

        madrigal_tec_DF.insert(loc=6, column="datetime", value=datetime_format_madrigal)


        '''
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        ##               Insert MLAT, MLON, MLT columns into madrigal TEC dataframe
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        '''
        # Another way to insert columns to the dataframe if position does not need to be specified - this will
        # automatically add the below columns to the end
        madrigal_tec_DF["mlat"] = 0.0
        madrigal_tec_DF["mlon"] = 0.0
        madrigal_tec_DF["mlt"] = 0.0
        madrigal_tec_DF

        year_loaded = madrigal_tec_DF.loc[:, "year"].values
        month_loaded = madrigal_tec_DF.loc[:, "month"].values
        day_loaded = madrigal_tec_DF.loc[:, "day"].values
        hour_loaded = madrigal_tec_DF.loc[:, "hour"].values
        min_loaded = madrigal_tec_DF.loc[:, "min"].values
        sec_loaded = madrigal_tec_DF.loc[:, "sec"].values
        lat_loaded = madrigal_tec_DF.loc[:, "gdlat"].values
        lon_loaded = madrigal_tec_DF.loc[:, "glon"].values

        datetime_loaded = madrigal_tec_DF.loc[:, "datetime"].values
        # converting datetime from 2013-11-07T00:02:30 to 2013-11-07 00:02:30 format
        datetime_loaded = pd.to_datetime(datetime_loaded)

        datetime_loaded_unique = np.unique(datetime_loaded)
        # converting datetime from 2013-11-07T00:02:30 to 2013-11-07 00:02:30 format
        datetime_loaded_unique = pd.to_datetime(datetime_loaded_unique)

        for dd in tqdm(range(0, len(datetime_loaded_unique)), leave=False, desc='Making Global TEC plots'):
            same_time_ind = np.where(datetime_loaded == datetime_loaded_unique[dd])[0]
            lat_loaded_array = lat_loaded[same_time_ind]
            lon_loaded_array = lon_loaded[same_time_ind]

            mlat, mlon, mlt = np.array(
                aacgmv2.get_aacgm_coord_arr(
                    lat_loaded_array, lon_loaded_array, 350, datetime_loaded_unique[dd]
                )
            )

            madrigal_tec_DF.loc[same_time_ind, "mlat"] = mlat
            madrigal_tec_DF.loc[same_time_ind, "mlon"] = mlon
            madrigal_tec_DF.loc[same_time_ind, "mlt"] = mlt



        '''
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        ##                                   Remove rows with nans
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        '''
        madrigal_tec_DF = madrigal_tec_DF.dropna(subset=["mlat", "mlt", "tec"])


        '''
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        # Getting required columns/data from all dataframes (madrigal, scin, etc)  to start
        # plotting TEC maps with scintillation indices
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        '''
        # columns from madrigal_tec_DF
        datetime_loaded = madrigal_tec_DF.loc[:, "datetime"].values
        mlat_loaded = madrigal_tec_DF.loc[:, "mlat"].values
        mlt_loaded = madrigal_tec_DF.loc[:, "mlt"].values
        glat_loaded = madrigal_tec_DF.loc[:, "gdlat"].values
        glon_loaded = madrigal_tec_DF.loc[:, "glon"].values
        tec_loaded = madrigal_tec_DF.loc[:, "tec"].values

        nan_ind = np.where(
            (np.isnan(mlat_loaded))
            | (np.isnan(mlt_loaded))
            | (np.isnan(tec_loaded))
        )[0]
        datetime_loaded = np.delete(datetime_loaded, nan_ind)
        mlat_loaded = np.delete(mlat_loaded, nan_ind)
        mlt_loaded = np.delete(mlt_loaded, nan_ind)
        glat_loaded = np.delete(glat_loaded, nan_ind)
        glon_loaded = np.delete(glon_loaded, nan_ind)
        tec_loaded = np.delete(tec_loaded, nan_ind)



        '''
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        ##                 Global TEC maps in geo (lat, lon) co-ordinates
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        '''
        ax1 = fig.add_subplot(ax0[tt])

        # ignoring pcolormesh cell edges warnings.
        warnings.filterwarnings("ignore", category=UserWarning)
        datetime_loaded_unique = np.unique(datetime_loaded)
        

        """
        # ¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        # iterating through timestamps in TEC map files
        # ¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        """
        req_time = pd.to_datetime(datetime_loaded_unique[0])

        '''
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        # Draw the coastlines of USA
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        '''
       
        
        # Inputting which labels to show
        if (tt==0) | (tt==1) | (tt==2):
            show_xticks = False
        else:
            show_xticks = True
            
        if (tt==0) | (tt==3):
            show_yticks = True
        else:
            show_yticks = False
        
      
        m = Basemap(projection='merc',llcrnrlat=20,urcrnrlat=60,\
                    llcrnrlon=-140,urcrnrlon=-50,lat_ts=30,resolution='i')
        m.drawcoastlines()
        m.drawcountries()
        # draw parallels and meridians.
        parallels = np.arange(-90.,91.,10)
        # Label the meridians and parallels
        m.drawparallels(parallels, labels=[show_yticks, False, False, False]) # labels=[left, right, top, bottom]
        # Draw Meridians and Labels
        meridians = np.arange(-180.,181.,20.)
        m.drawmeridians(meridians, labels=[False, False, False, show_xticks]) # labels=[left, right, top, bottom]
        m.drawmapboundary(fill_color='white')
    


        '''
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        # This is the step that transforms the data into the map's projection
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        '''
        lon_proj, lat_proj = m(glon_loaded, glat_loaded)  

        '''
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        # Plot TEC on the trnasfored (lat,lon) co-ordinates
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        '''
        img_tec = ax1.scatter(lon_proj, lat_proj, c=tec_loaded, 
                         vmin=0, vmax =40, cmap='turbo', 
                         s=8, edgecolors='none')


        '''
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        # Show xlabels only for the bottom plots
        # Also put labels for the left and bottom plots
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        '''
        if (tt==0) | (tt == 3):
            ax1.set_ylabel(r'Geographic latitude ($^\circ$)', labelpad=70)
            
        if (tt==3) | (tt == 4) | (tt==5):
            ax1.set_xlabel(r'Geographic longitude ($^\circ$)', labelpad=40)
            
        
        
        '''
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        # Include the datetime as title
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        '''
        date_as_title = month_string_here + ' ' + day_here + ', ' + year_here
        time_as_title = req_time.strftime("%H:%M:%S")
        
        # datetime_as_title = date_as_title + "\n" + time_as_title + "\n"
        fig.suptitle(date_as_title, y=0.91, fontweight="bold", fontsize=common_fnt_size+5)
        ax1.set_title(
            " ".join(time_as_title), fontweight="bold", fontsize=common_fnt_size, loc="center"
        )
        '''
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        # Include the colorbar
        #¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤¤
        '''
        # Create a common colorbar on the right
        if tt == len(required_datetimes)-1:
            
            cbar_ax = fig.add_axes([0.91, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
            cbar = fig.colorbar(img_tec, cax=cbar_ax) 
            cbar.set_label('VTEC (TECU)') 
             
                    
    else:
        continue
            


